# 案例 09 · 大 key 撑爆 Redis，榜单缓存写入被拒

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

全站榜单被写成单个 6MB key，分类榜单 10×2MB，`maxmemory=32MB` 且策略为 `noeviction`，内存触顶后所有写命令报 OOM，首页榜单降级为空。

## 练习目标

- 用 `INFO memory` 同时看 used/max/policy/evicted
- 用 `--bigkeys` 找到大 key
- 切淘汰策略止血 + 拆分/删除大 key
- 用 `redis_mem_guard.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-09-redis")
lab.reset(start_api=True)
lab.start_redis()
lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_redis_memory()
print(lab.sh("redis-cli INFO memory"))


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**内存与策略**

In [ ]:
print(lab.sh("redis-cli INFO memory"))

**大 key 扫描**

In [ ]:
print(lab.sh("redis-cli --bigkeys"))

**写入复现**

In [ ]:
print(lab.sh("redis-cli SET feed:rank:probe 1048576"))

**应用日志**

In [ ]:
print(lab.sh("tail -n 3 /var/log/xhs/note-api.log"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
info = {}
for line in lab.sh("redis-cli INFO memory").splitlines():
    if ":" in line and not line.startswith("#"):
        k, _, v = line.partition(":")
        info[k.strip()] = v.strip()
used, maxmem = int(info["used_memory"]), int(info["maxmemory"])
print(f"used={used / 1048576:.1f}MB max={maxmem / 1048576:.1f}MB policy={info['maxmemory_policy']}")
print("判断：used ≈ max 且策略为 noeviction → 写入被拒，需要先恢复写入能力")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
# 1) 切淘汰策略，秒级恢复写入（可回滚）
lab.sh("redis-cli CONFIG SET maxmemory-policy allkeys-lru")
# 2) 删除/拆分超大 key（榜单可重算）
lab.sh("redis-cli DEL feed:rank:global:all")
lab.sh("redis-cli DEL note:detail:8f2a41c9")
lab.redis("SET feed:rank:global:all 2097152")
print(lab.sh("redis-cli INFO memory"))
print("写入验证:", lab.sh("redis-cli SET feed:rank:smoke 1048576"))


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('写入恢复', bool('OK' in lab.sh('redis-cli SET feed:rank:smoke 1048576')))
lab.check('内存低于上限', bool(int([l.split(':')[1] for l in lab.sh('redis-cli INFO memory').splitlines() if l.startswith('used_memory:')][0]) <= int([l.split(':')[1] for l in lab.sh('redis-cli INFO memory').splitlines() if l.startswith('maxmemory:')][0])))

## 步骤 6 · 自动化沉淀

脚本读取 `INFO memory` 与 `--bigkeys`，切 `allkeys-lru`、删除超大 key，最后用 `SET` 验证写入恢复且内存低于上限。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.start_redis()
lab.inject_redis_memory()
out = lab.auto('redis_mem_guard.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. `noeviction`、`volatile-lru`、`allkeys-lru` 分别适合什么业务？
2. 大 key 除了内存，还有哪些运行时风险？
3. 缓存被淘汰后如何避免请求穿透到 DB？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')